# Random Forest: Train and Compare

**Motivation.** The decision tree was the first model to beat persistence on 2023 (0.658 against 0.651), but it is fragile:

- its 2023 score **jumped by about 0.01** between neighbouring settings, more than its margin over persistence;
- over **five years** it only **tied** with persistence (mean −0.003, ahead in 3 of 5 years).

A random forest reduces that variance by voting over many decorrelated trees. Each tree is trained on a bootstrap sample of the days (**bagging**; about a third of the days are left out of each sample), and each split chooses from a random subset of the features (`max_features`), so the 200 trees do not all split first on `aqi_today` and make the same errors.

**The bar** (set in `dt_train.ipynb`, section 9): a model only counts as better if it beats persistence **on average over the five years 2019–2023 and in most of them**, not just in 2023.

Decisions so far: [`docs/model_decisions.md`](../docs/model_decisions.md). Run `lr_train_baseline.ipynb` and `dt_train.ipynb` first: this notebook loads `models/lr_best.bin` and `models/dt_best.bin` to compare with them.

## 1. Setup

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

import sys
sys.path.insert(0, '../scripts')                 # utility.py lives in scripts/
from utility import (AQI_CATEGORIES, prepare_dataset, split_train_val_test, add_lr_features,
                     forecast_scorecard, class_scores_table, rolling_validation, rolling_summary)

DATA_PATH = '../data/processed/aqi_weather_2015_2025.csv'
MODEL_DIR = Path('models')
TARGET = 'tomorrow_status'
TREND = ['aqi_roll7_mean', 'aqi_roll7_std', 'aqi_diff1', 'aqi_dev7']
pd.set_option('display.max_columns', 30)

## 2. The same days as every other model

The same data pipeline, the same 195 rows dropped and the same time split, so every model is scored on exactly the same 2023 days.

In [ ]:
df = prepare_dataset(DATA_PATH)
before = len(df)
df = df[df[TREND].notna().all(axis=1)]                   # the same 195 rows every model drops
df_train, df_val, df_test = split_train_val_test(df)

print(f'Dropped {before - len(df)} rows with empty trend features')
print('train:', len(df_train), '| validation:', len(df_val), '| test:', len(df_test))

## 3. Features: the same raw columns as the decision tree

No transforms (no scaling, no log, no month on a circle, no pressure relative to the city; reasons in `dt_train.ipynb`, section 3). The features are exactly the decision tree's: all raw weather columns, all four trend features, the pollutant flags and `city`.

In [ ]:
TREE_NUMERIC = (['aqi_today'] + TREND +
                ['temp_mean', 'temp_max', 'temp_min', 'temp_range', 'humidity_mean', 'dew_point_mean',
                 'precip_sum', 'pressure_mean', 'cloud_cover_mean', 'wind_speed_mean', 'wind_speed_max',
                 'wind_u_mean', 'wind_v_mean', 'month'])
TREE_BINARY = [c for c in df.columns if c.startswith('has_')] + ['aqi_imputed']
TREE_CATEGORICAL = ['city']
FEATURES = TREE_NUMERIC + TREE_BINARY + TREE_CATEGORICAL

def to_records(part):
    return part[FEATURES].to_dict(orient='records')

X_train, X_val = to_records(df_train), to_records(df_val)
y_train = df_train[TARGET].astype(str).to_numpy()
y_val = df_val[TARGET].astype(str).to_numpy()

assert df_train[FEATURES].notna().all().all() and df_val[FEATURES].notna().all().all(), 'missing values in the features'
print(f'{len(TREE_NUMERIC)} numeric + {len(TREE_BINARY)} yes/no + {len(TREE_CATEGORICAL)} text = {len(FEATURES)} features')

## 4. The settings of a forest

| Setting | Meaning | Effect |
|---|---|---|
| `n_estimators` | Number of trees | More trees → a steadier vote. Does not overfit; only costs time |
| `max_features` | Features available at each split, out of the 40 columns after one-hot encoding the city (`'sqrt'` = √40 ≈ 6, `0.33` = a third, `1.0` = all) | Fewer → more decorrelated trees whose errors cancel better; too few → weak trees |
| `min_samples_leaf` | Minimum training days per leaf | As in the single tree, but a forest tolerates much smaller leaves (section 5) |
| `class_weight` | `None`, or `'balanced_subsample'`: rare categories weighted up, recomputed on each tree's own sample | Balanced lowers the bar for the rare categories (Very Poor, Severe) |

`n_jobs=-1` trains on all processor cores; `random_state=1` fixes the random samples across runs.

In [ ]:
def make_forest(min_samples_leaf=1, max_features='sqrt', class_weight=None, n_estimators=200):
    return Pipeline([
        ('dv', DictVectorizer(sparse=False)),             # city -> one-hot; numbers pass through
        ('forest', RandomForestClassifier(n_estimators=n_estimators, min_samples_leaf=min_samples_leaf,
                                          max_features=max_features, class_weight=class_weight,
                                          n_jobs=-1, random_state=1)),
    ])

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=AQI_CATEGORIES, average='macro')

## 5. Two quick experiments

### 5.1 Effect of averaging trees on overfitting

In `dt_train.ipynb` an unconstrained tree overfit the training years: **1.000** on training, only **0.513** on 2023. Below, 200 such trees (no depth limit, a leaf may hold a single day) are combined by vote:

In [ ]:
free_forest = make_forest(min_samples_leaf=1).fit(X_train, y_train)

print(f'Macro F1 on training years: {macro_f1(y_train, free_forest.predict(X_train)):.3f}')
print(f'Macro F1 on 2023:           {macro_f1(y_val, free_forest.predict(X_val)):.3f}   '
      f'(one unlimited tree 0.513, persistence 0.651)')

**Voting removes most of the overfitting.** Each of the 200 trees fits its sample perfectly (training score **1.000**, as for the single unconstrained tree), but on 2023 the forest scores **0.670** instead of 0.513, already above persistence: the trees overfit to *different* noise (different days and features), which cancels in the vote.

**For a forest, a large train/validation gap is therefore not a warning sign by itself**; what matters is the score on unseen days and, later, on five years.

### 5.2 Number of trees

The same forest (at least 5 days per leaf) with 1 to 500 trees, scored on 2023:

In [ ]:
TREE_COUNTS = [1, 5, 10, 25, 50, 100, 200, 300, 500]
trees_curve = pd.Series({n: macro_f1(y_val, make_forest(5, n_estimators=n).fit(X_train, y_train).predict(X_val))
                         for n in TREE_COUNTS}, name='macro F1 on 2023')

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(trees_curve.index, trees_curve.values, marker='o', label='random forest')
ax.axhline(0.651, color='grey', linestyle='--', linewidth=1, label='persistence (0.651)')
ax.set_xscale('log')
ax.set_xticks(TREE_COUNTS, [str(n) for n in TREE_COUNTS])
ax.set_xlabel('number of trees (log scale)')
ax.set_ylabel('macro F1 on 2023')
ax.legend()
plt.tight_layout()
plt.show()
trees_curve.round(3).to_frame().T

**Findings:**
- **One tree is weak** (0.528): the same overfitting, made worse by training on a sample of the days.
- **The score rises quickly** to 0.658 with 25 trees, passing persistence.
- **Flat from about 100 trees** (0.670–0.677). The variation (0.677 at 200, 0.671 at 300) is sampling noise, not a real difference.
- **More trees never hurt**, they only cost time. The forests below use **200 trees**.

## 6. Tuning on 2023

Every combination of 4 `max_features` × 4 leaf sizes × 2 class weightings = 32 forests of 200 trees, each scored on the training years and on 2023 (about 3 minutes).

In [ ]:
MAX_FEATURES = ['sqrt', 0.33, 0.5, 1.0]                  # 'sqrt' = about 6 of the 40 columns; 1.0 = all of them
LEAVES = [1, 5, 20, 50]                                  # minimum days in every leaf
WEIGHTS = {'no class weights': None, 'balanced': 'balanced_subsample'}

rows = []
for weight_name, class_weight in WEIGHTS.items():
    for max_features in MAX_FEATURES:
        for leaf in LEAVES:
            m = make_forest(leaf, max_features, class_weight).fit(X_train, y_train)
            pred_va = m.predict(X_val)
            rows.append({
                'weights': weight_name, 'max_features': max_features, 'min_samples_leaf': leaf,
                'train_macro_f1': macro_f1(y_train, m.predict(X_train)),
                'val_macro_f1': macro_f1(y_val, pred_va),
                'said_severe': (pred_va == 'Severe').sum(),
            })

tuning = pd.DataFrame(rows)
tuning['gap'] = tuning['train_macro_f1'] - tuning['val_macro_f1']
print(len(tuning), 'forests trained | real Severe days in 2023:', (y_val == 'Severe').sum())
tuning.sort_values('val_macro_f1', ascending=False).head(10).round(3)

**Macro F1 on 2023** for every combination (rows: weighting and `max_features`, columns: minimum days per leaf):

In [ ]:
(tuning.pivot_table(index=['weights', 'max_features'], columns='min_samples_leaf', values='val_macro_f1')
       .reindex(['no class weights', 'balanced'], level=0)
       .round(3))

**Findings:**
- **Almost every forest beats persistence on 2023.** All 24 forests with leaves of 1 to 20 days score 0.652 to 0.679, against persistence's 0.651. Only 4 of the single tree's 60 settings got above 0.651, all with exactly 100 days per leaf.
- **The top is flat.** The 10 best forests are within **0.005** of each other (0.674–0.679), less than the year-to-year noise seen in rolling validation, so **2023 alone cannot pick the best setting**.
- **Leaves of 50 days are too coarse,** especially with balanced weights (0.632–0.642).
- **Balanced weights tie with no weights at the top** (both 0.679), but predict "Severe" much more often (66 times against 40, with 51 real Severe days).

Section 7 uses five years to choose among the top settings.

## 7. Let five years choose

Shortlist: the **3 best settings of each weighting** on 2023. Each goes through rolling validation (train on the years before, score one year, for 2019 to 2023), with the decision tree from `dt_best.bin` as reference. The test years 2024–2025 stay locked.

`rolling_summary` in `utility.py` applies the bar: `passes_bar` is true only if a model beats persistence on the five-year average **and** in most of the years.

**Selection rule:** among the forests that pass, keep those within 0.002 of the best five-year average (smaller differences are noise). Of those, prefer more years ahead of persistence, then the smaller training gap.

In [ ]:
with open(MODEL_DIR / 'dt_best.bin', 'rb') as f:        # saved by dt_train.ipynb, section 10
    dt_bundle = pickle.load(f)
with open(MODEL_DIR / 'lr_best.bin', 'rb') as f:        # saved by lr_train_baseline.ipynb
    lr_bundle = pickle.load(f)

SHORTLIST = (tuning.sort_values('val_macro_f1', ascending=False)
                   .groupby('weights', sort=False).head(3)
                   .copy())
SHORTLIST['name'] = SHORTLIST.apply(
    lambda r: f"forest: {r['weights']}, max_features {r['max_features']}, leaf {r['min_samples_leaf']}", axis=1)

def forest_maker(row):                                   # a function that returns a fresh, unfitted forest
    return lambda: make_forest(row['min_samples_leaf'], row['max_features'], WEIGHTS[row['weights']])

def make_dt():                                           # the chosen decision tree, same settings as dt_best.bin
    return Pipeline([('dv', DictVectorizer(sparse=False)),
                     ('tree', DecisionTreeClassifier(**dt_bundle['params'], random_state=1))])

def tree_xy(train, val):
    return to_records(train), to_records(val)

CANDIDATES = {row['name']: forest_maker(row) for _, row in SHORTLIST.iterrows()}
CANDIDATES['decision tree (dt_best)'] = make_dt

df_dev = pd.concat([df_train, df_val])                   # 2015-2023 only; the test years stay locked
rolling = pd.concat({name: rolling_validation(df_dev, make, tree_xy) for name, make in CANDIDATES.items()},
                    names=['model']).reset_index(level=0)
SHORTLIST[['name', 'val_macro_f1', 'said_severe']].round(3)

**Macro F1 per year** (persistence first, for reference):

In [ ]:
by_year = rolling.pivot_table(index='model', columns='val_year', values='macro_f1')
persistence_row = rolling.groupby('val_year')['persistence_macro_f1'].first().rename('persistence')
by_year = pd.concat([by_year, persistence_row.to_frame().T]).reindex(['persistence'] + list(CANDIDATES))
by_year.round(3)

**Summary over the five years** (`spread` = how much the score moves from year to year; `mean_train_gap` = training score minus validation score):

In [ ]:
five_years = rolling_summary(rolling)
five_years.round(3)

In [ ]:
passing = five_years[five_years.index.str.startswith('forest') & five_years['passes_bar']]
assert len(passing), 'no forest passes the bar: keep persistence'
close = passing[passing['mean_macro_f1'] >= passing['mean_macro_f1'].max() - 0.002]    # within 0.002 = a tie
chosen_name = close.sort_values(['years_won', 'mean_train_gap'], ascending=[False, True]).index[0]
chosen = SHORTLIST.set_index('name').loc[chosen_name]

print(f"Chosen: {chosen.name}")
print(f"  five-year mean {five_years.loc[chosen.name, 'mean_macro_f1']:.3f} "
      f"(persistence {by_year.loc['persistence'].mean():.3f}), "
      f"ahead in {five_years.loc[chosen.name, 'years_won']} of {five_years.loc[chosen.name, 'years']} years")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for name in ['persistence', 'decision tree (dt_best)', chosen.name]:
    style = dict(color='grey', linestyle='--') if name == 'persistence' else {}
    ax.plot(by_year.columns, by_year.loc[name], marker='o', label=name, **style)
ax.set_xticks(list(by_year.columns))
ax.set_xlabel('validation year (learned from all earlier years)')
ax.set_ylabel('macro F1')
ax.set_title('Rolling validation: random forest vs decision tree vs persistence')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Findings:**
- **The random forest is the first model to pass the bar.** All 6 shortlisted forests beat persistence on the five-year average (+0.009 to +0.030), in at least 4 of the 5 years. The decision tree, run the same way, does not (−0.003, 3 of 5 years).
- **Chosen forest** (balanced weights, `max_features` 0.5, at least 5 days per leaf): **0.664** against persistence's **0.634** (+0.030), **ahead in all 5 years**, including **2022** (+0.020), the year every earlier model lost. It also has the smallest year-to-year spread (0.027).
- **Five years changed the class-weight decision.** On 2023, balanced and no weights tied (0.679 each). Over five years, balanced wins by **+0.012** (0.664 against 0.652), mostly in the harder years 2019 and 2022 (2022: 0.639 against 0.598–0.608).
- **Likely reason balanced weights help here but hurt logistic regression and the single tree:** in logistic regression they moved the bar for "Severe" on *every* day; in a forest they only shift the votes inside each tree's small leaves, which 200 trees then average, so the rare categories gain weight without a flood of rare forecasts. **Class weighting has to be re-checked for every model type, on five years.**
- **The training gap is large** (0.23), but as section 5.1 showed, that is expected for a forest; the five-year scores show it generalises.

## 8. Compare on 2023: persistence vs logistic regression vs decision tree vs random forest

The chosen forest is retrained on 2015–2022 and scored on 2023 with the same scorecard as every other model. Logistic regression and the decision tree are loaded from their saved bundles.

In [ ]:
forest = make_forest(chosen['min_samples_leaf'], chosen['max_features'], WEIGHTS[chosen['weights']]).fit(X_train, y_train)
pred_forest = forest.predict(X_val)

df_val_lr = add_lr_features(df_val, pd.Series(lr_bundle['city_pressure']))   # the LR needs its own transforms
lr_features = sum(lr_bundle['features'].values(), [])
pred_lr = lr_bundle['model'].predict(df_val_lr[lr_features].to_dict(orient='records'))
pred_tree = dt_bundle['model'].predict(X_val)

PREDICTIONS = {
    'persistence': df_val['status_today'].astype(str).to_numpy(),
    'logistic regression': pred_lr,
    'decision tree': pred_tree,
    'random forest': pred_forest,
}
scorecard = forecast_scorecard(y_val, PREDICTIONS, y_today=df_val['status_today'])
scorecard

**Precision, recall and F1 for every category:**

In [ ]:
class_scores_table(y_val, PREDICTIONS).round(2)

**Confusion matrix of the forest** (rows: what really happened, columns: what the forest forecast):

In [ ]:
pd.DataFrame(confusion_matrix(y_val, pred_forest, labels=AQI_CATEGORIES),
             index=pd.Index(AQI_CATEGORIES, name='actual'),
             columns=pd.Index(AQI_CATEGORIES, name='predicted'))

**Findings:**

| 2023 | Persistence | Logistic regression | Decision tree | Random forest |
|---|---|---|---|---|
| Macro F1 | 0.651 | 0.647 | 0.658 | **0.679** |
| Accuracy | 0.710 | 0.729 | 0.723 | **0.734** |
| Severe caught / missed / false alarms | 27 / 24 / 25 | 20 / 31 / **11** | 25 / 26 / 19 | **31 / 20** / 35 |
| Dangerous recall / precision | 0.749 / 0.753 | 0.807 / **0.761** | 0.768 / 0.740 | **0.840** / 0.760 |
| Change days right | 0% | **35.3%** | 27.8% | 26.8% |
| Unchanged days right | **100%** | 88.3% | 90.4% | 92.4% |

- **Best model on 2023 on the main scores:** macro F1 0.679 (+0.028 over persistence), accuracy 73.4%.
- **Best Dangerous alert so far:** warns on **84%** of Very Poor and Severe days (logistic regression 81%, persistence 75%) at **76%** precision, matching logistic regression.
- **Most Severe days caught (31 of 51), but also the most Severe false alarms (35):** Severe precision is only 47%, the cost of balanced weights. Per the confusion matrix, **30 of the 35 false alarms were Very Poor days**, also dangerous; only 5 were Poor days.
- **Better at the rare ends:** Good F1 0.66 and Poor F1 0.63, the best of all four models.
- **Change days remain hard:** 26.8% right, below logistic regression (35.3%). The gain comes from unchanged days (92.4%) and the rare categories.

## 9. Feature importance

The forest's feature importance next to the single tree's: each feature's share of the total reduction in mistakes; for the forest, averaged over its 200 trees.

In [ ]:
importance = pd.DataFrame({
    'random forest': pd.Series(forest['forest'].feature_importances_, index=forest['dv'].feature_names_),
    'decision tree': pd.Series(dt_bundle['model']['tree'].feature_importances_,
                               index=dt_bundle['model']['dv'].feature_names_),
})
importance.sort_values('random forest', ascending=False).head(12).round(3)

**Findings:**
- **`aqi_today` still matters most, but with 37% of the importance instead of 83%.** With `max_features` 0.5, `aqi_today` is unavailable at about half of the splits, so the trees also learn the next-best signals: the **weekly average** (`aqi_roll7_mean`, 17%), the **daily temperature range** (`temp_range`, 5%; a large range often indicates clear, calm nights that trap pollution near the ground), **wind** and **humidity**.
- **Importance is not causation.** Correlated features (the four temperature columns, the trend features) share their importance, so a small value does not mean a feature is useless.

## 10. Save the model

In [ ]:
MODEL_PATH = MODEL_DIR / 'rf_best.bin'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
bundle = {
    'model': forest,                                       # DictVectorizer + random forest
    'params': {'n_estimators': 200, 'max_features': chosen['max_features'],
               'min_samples_leaf': int(chosen['min_samples_leaf']), 'class_weight': WEIGHTS[chosen['weights']]},
    'features': {'numeric': TREE_NUMERIC, 'binary': TREE_BINARY, 'categorical': TREE_CATEGORICAL},
    'categories': AQI_CATEGORIES,
    'cities': sorted(df_train['city'].unique()),           # the service must reject any other city
    'validation': scorecard['random forest'].to_dict(),
    'five_years': five_years.loc[chosen.name].to_dict(),
    'tuning': tuning.round(4).to_dict(orient='records'),
}
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(bundle, f)
with open(MODEL_PATH, 'rb') as f:
    assert (pickle.load(f)['model'].predict(X_val) == pred_forest).all()
print(f'Saved and verified: {MODEL_PATH.resolve()} ({MODEL_PATH.stat().st_size / 1e6:.0f} MB)')

## Discussion

1. **The random forest is the first model to pass the bar.** Over 2019–2023 it averages **0.664** against persistence's **0.634**, ahead in **all five years**, including 2022, where every earlier model lost.
2. **Voting fixes the single tree's main weakness.** 200 unconstrained trees fit the training data as fully as one (training 1.000), but their vote scores 0.670 on 2023 instead of 0.513. Almost every forest setting beats persistence on 2023, against 4 of 60 single-tree settings.
3. **One year could not choose the setting; five years could.** The 10 best forests on 2023 were within 0.005. Over five years, balanced class weights were clearly better (+0.012), which 2023 alone would have missed.
4. **On 2023 the forest is best on macro F1 (0.679), accuracy (73.4%) and the Dangerous alert (84% caught, 76% right).** The trade-off is Severe: the most Severe days caught (31 of 51), but also the most Severe false alarms (35, of which 30 were Very Poor days). The planned **Severe threshold** step can tune that balance.
5. **Cost: a 114 MB model file**, above GitHub's 100 MB limit, so `experiments/models/*.bin` should not be committed. `train.py` will rebuild the model from the data.

**Next: XGBoost** (`xgb_train.ipynb`), held to the same bar and compared with this forest's five-year average of 0.664.